## Check broken logs

In [ ]:
import re
def find_max_ckpt(names):
    steps = []
    for name in names:
        m = re.fullmatch(r'checkpoint-(\d+)', name)
        if m:
            steps.append(int(m.group(1)))
    return max(steps)

In [ ]:
import os
cur_path = '.'
proj_folders = os.listdir()
for proj_folder in proj_folders:
    proj_folder_path=os.path.join(cur_path, proj_folder)
    if os.path.isdir(proj_folder_path):
        for run_folder in os.listdir(proj_folder_path):
            # print("inspecting", run_folder, "...")
            run_folder_path = os.path.join(proj_folder_path, run_folder)
            try:
                ckpt_list = os.listdir(run_folder_path)
                if 'eval_results.csv' in ckpt_list:
                    ckpt_list.remove('eval_results.csv')
                if 'README.md' in ckpt_list:
                    ckpt_list.remove('README.md')
                max_ckpt = find_max_ckpt(ckpt_list)
                ckpt_path = os.path.join(run_folder_path, 'checkpoint-'+str(max_ckpt), 'trainer_state.json')
                with open(ckpt_path, 'r') as f:
                    data = json.load(f)
                    if data['log_history'][0]['step'] != 10:
                        print('Run ', run_folder_path, 'has broken logs! Broken point: ', data['log_history'][0]['step'])
                    else:
                        continue
            except:
                continue



# Find best ckpt and save


In [ ]:
file = "./qwen2.5-0.5B_GSM8K_nov_30_reward_rectified/-shared-public-models-Qwen-Qwen2.5-0.5B-Instruct-GSM8K-kl-0.001-DAPO/checkpoint-1244/trainer_state.json"
with open(file, 'r') as f:
    data = json.load(f)
data['log_history'][5]

In [ ]:
def find_best_reward_ckpt(file, threshold):
    metric='eval_rewards/correctness_reward_func/mean'
    best_value = -1e06
    best_step = 0
    with open(ckpt_path, 'r') as f:
        data = json.load(f)
    for item in data['log_history']:
        if threshold and item['step'] > threshold: # prevent the DAPO-Math-17k constraint we set for 1.5B models since we only allow training for one epoch.
            break

        if metric in item and item['step'] % 100 == 0:
            if item[metric] > best_value:
                best_step = item['step']
                best_value = item[metric]
    return best_step

import os
cur_path = '.'
proj_folders = os.listdir()
for proj_folder in proj_folders:
    proj_folder_path=os.path.join(cur_path, proj_folder)
    if os.path.isdir(proj_folder_path):
        for run_folder in os.listdir(proj_folder_path):
            # print("inspecting", run_folder, "...")
            run_folder_path = os.path.join(proj_folder_path, run_folder)
            try:
                ckpt_list = os.listdir(run_folder_path)
                if 'eval_results.csv' in ckpt_list:
                    ckpt_list.remove('eval_results.csv')
                if 'README.md' in ckpt_list:
                    ckpt_list.remove('README.md')
                max_ckpt = find_max_ckpt(ckpt_list)
                ckpt_path = os.path.join(run_folder_path, 'checkpoint-'+str(max_ckpt), 'trainer_state.json')
                if '1.5B' in run_folder_path and 'DAPO-Math-17k' in run_folder_path:
                    threshold=119400
                elif '7B' in run_folder_path and 'DAPO-Math-17k' in run_folder_path:
                    threshold=10000
                else: 
                    threshold=None
                best_step = find_best_reward_ckpt(ckpt_path, threshold)
                # TODO:open a txt file and save under run_folder_path the best step
                out_path = os.path.join(run_folder_path, 'best_step.txt')
                with open(out_path, 'w') as f:
                    f.write(f"{best_step}")
            except:
                continue

In [ ]:
# path="./Qwen2.5-0.5B-Instruct_GSM8K_G2RPO_next_token/-shared-public-models-Qwen-Qwen2.5-0.5B-Instruct-GSM8K-kl-0.001-G2RPO-surp-1e-06-conf-1e-06-horizon-1-decay-0.5"
# with open(os.path.join(path, 'best_step.txt'), 'r') as f:
#     ckpt_numbers=[int(line.strip()) for line in f.readlines()]
# print(ckpt_numbers)


## Delete all eval_results.csv

In [ ]:
import os
cur_path = '.'
proj_folders = os.listdir()
for proj_folder in proj_folders:
    proj_folder_path=os.path.join(cur_path, proj_folder)
    if os.path.isdir(proj_folder_path):
        for run_folder in os.listdir(proj_folder_path):
            # print("inspecting", run_folder, "...")
            run_folder_path = os.path.join(proj_folder_path, run_folder)
            try:
                ckpt_list = os.listdir(run_folder_path)
                if 'eval_results.csv' in ckpt_list:
                    os.remove(os.path.join(run_folder_path, 'eval_results.csv'))
            except:
                continue

In [ ]:
# import os
# cur_path = '.'
# proj_folders = os.listdir()
# for proj_folder in proj_folders:
#     proj_folder_path=os.path.join(cur_path, proj_folder)
#     if os.path.isdir(proj_folder_path):
#         # print(proj_folder_path)
#         if proj_folder in ['Qwen2.5-0.5B-Instruct_GSM8K','Qwen2.5-0.5B-Instruct_MATH-500', 'Qwen2.5-0.5B-Instruct_DAPO-Math-17k', 'Qwen2.5-1.5B-Instruct_GSM8K', 'Qwen2.5-1.5B-Instruct_MATH-500', 'Qwen2.5-1.5B-Instruct_DAPO-Math-17k']:
#             print("inspecting", proj_folder_path, "...")
#             for run_folder in os.listdir(proj_folder_path):
#                 # print("inspecting", run_folder, "...")
#                 run_folder_path = os.path.join(proj_folder_path, run_folder)
#                 try:
#                     ckpt_list = os.listdir(run_folder_path)
#                     if 'eval_results.csv' in ckpt_list:
#                         # os.remove(os.path.join(run_folder_path, 'eval_results.csv'))
#                         old_path = os.path.join(run_folder_path, 'eval_results.csv')
#                         new_path = os.path.join(run_folder_path, 'old_eval_results.csv')
#                         os.rename(old_path, new_path)
#                 except:
#                     continue

## Collect New Eval Results

In [ ]:
from pathlib import Path
import csv

output = Path("eval_results_unified.csv")
rows = []

for csv_path in Path(".").glob("*/*/eval_results.csv"):  # pattern: */X/eval_results.csv
    first_level_raw = csv_path.parent.parent.name
    first_level = (
        "next_token" if "next_token" in first_level_raw
        else "last_token" if "last_token" in first_level_raw
        else "None"
    )
    second_level = csv_path.parent.name  # the X in */X/...
    last_row = None
    with csv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if row and any(cell.strip() for cell in row):  # skip empty lines
                last_row = row
    if last_row is not None:
        rows.append([first_level, second_level] + last_row)

with output.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print(f"Collected {len(rows)} rows into {output}")

In [ ]:
from pathlib import Path
import csv

output = Path("eval_results_unified_baselines_feb21.csv")
rows = []

for csv_path in Path(".").glob("*/*/eval_results.csv"):  # pattern: */X/eval_results.csv
    first_level_raw = csv_path.parent.parent.name
    if first_level_raw in ['Qwen2.5-0.5B-Instruct_GSM8K','Qwen2.5-0.5B-Instruct_MATH-500', 'Qwen2.5-0.5B-Instruct_DAPO-Math-17k', 'Qwen2.5-1.5B-Instruct_GSM8K', 'Qwen2.5-1.5B-Instruct_MATH-500', 'Qwen2.5-1.5B-Instruct_DAPO-Math-17k']:
        model, data = first_level_raw.split('_')
        baseline = csv_path.parent.name.split('-')[-1]  # the X in */X/...
        last_row = None
        with csv_path.open(newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                if row and any(cell.strip() for cell in row):  # skip empty lines
                    last_row = row
        if last_row is not None:
            rows.append([model, data, baseline] + last_row)

with output.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print(f"Collected {len(rows)} rows into {output}")

In [ ]:
from pathlib import Path
import csv
import re

def parse_instruct_tag(s: str):
    m = re.search(r"-shared-public-models-Qwen-(.+?-Instruct)-(.+?)-kl-0.001-([^-]+)", s)
    if not m:
        raise ValueError("Tag is not in expected Instruct format")
    model, data, method = m.groups()
    return [model, data, method]

output = Path("eval_results_unified.csv")
rows = []

for csv_path in Path(".").glob("*/*/eval_results.csv"):  # */X/eval_results.csv
    first_level_raw = csv_path.parent.parent.name
    first_level = (
        "next_token" if "next_token" in first_level_raw
        else "last_token" if "last_token" in first_level_raw
        else "None"
    )
    second_level = csv_path.parent.name

    last_row = None
    with csv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if row and any(cell.strip() for cell in row):
                last_row = row
    if last_row is None:
        continue

    ident = second_level
    try:
        print(ident)
        model, dataset, baseline = parse_instruct_tag(ident)
    except ValueError:
        model = dataset = baseline = None

    remainder = last_row[:1] + last_row[2:]  # keep col0, drop col1 (identifier)
    rows.append([first_level, second_level, model, dataset, baseline] + remainder)

with output.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print(f"Collected {len(rows)} rows into {output}")

## Janurary 2: Find top 5 ckpts for each run and pick the one with best token efficiency.

In [ ]:
# rename old eval_results.csv
import os
cur_path = '.'
proj_folders = os.listdir()
for proj_folder in proj_folders:
    proj_folder_path=os.path.join(cur_path, proj_folder)
    if os.path.isdir(proj_folder_path):
        for run_folder in os.listdir(proj_folder_path):
            # print("inspecting", run_folder, "...")
            run_folder_path = os.path.join(proj_folder_path, run_folder)
            try:
                ckpt_list = os.listdir(run_folder_path)
                if 'eval_results.csv' in ckpt_list:
                    os.rename(os.path.join(run_folder_path, 'eval_results.csv'))
            except:
                continue

In [ ]:
import numpy as np
import json
def find_best_reward_ckpt(file, threshold):
    metric='eval_rewards/correctness_reward_func/mean'
    best_value = -1e06
    best_step = 0
    reward_list = []
    with open(file, 'r') as f:
        data = json.load(f)
    for item in data['log_history']:
        if threshold and item['step'] > threshold: # prevent the DAPO-Math-17k constraint we set for 1.5B models since we only allow training for one epoch.
            break
        if metric in item and item['step'] % 100 == 0:
            reward_list.append(item[metric])
    best_steps = list((np.argsort(np.array(reward_list))[-5:][::-1] + 1) * 100)
    return best_steps

import os
cur_path = '.'
proj_folders = os.listdir()
for proj_folder in proj_folders:
    proj_folder_path=os.path.join(cur_path, proj_folder)
    if os.path.isdir(proj_folder_path):
        for run_folder in os.listdir(proj_folder_path):
            # print("inspecting", run_folder, "...")
            run_folder_path = os.path.join(proj_folder_path, run_folder)
            try:
                ckpt_list = os.listdir(run_folder_path)
                print(ckpt_list)
                if 'eval_results.csv' in ckpt_list:
                    ckpt_list.remove('eval_results.csv')
                if 'README.md' in ckpt_list:
                    ckpt_list.remove('README.md')
                max_ckpt = find_max_ckpt(ckpt_list)
                print('max_ckpt', max_ckpt)
                ckpt_path = os.path.join(run_folder_path, 'checkpoint-'+str(max_ckpt), 'trainer_state.json')
                # if '1.5B' in run_folder_path and 'DAPO-Math-17k' in run_folder_path:
                #     threshold=119400
                # elif '7B' in run_folder_path and 'DAPO-Math-17k' in run_folder_path:
                #     threshold=10000
                if 'DAPO-Math-17k' in run_folder_path:
                    threshold=10000
                else: 
                    threshold=None
                best_steps = find_best_reward_ckpt(ckpt_path, threshold)
                # TODO:open a txt file and save under run_folder_path the best step
                out_path = os.path.join(run_folder_path, 'best_steps.txt')
                with open(out_path, 'w') as f:
                    # each row one number
                    for num in best_steps:
                        f.write(f"{num}\n")
            except:
                continue

In [ ]:
best_steps_file = "./Qwen2.5-7B-Instruct_GSM8K_G2RPO_last_token/-shared-public-models-Qwen-Qwen2.5-7B-Instruct-GSM8K-kl-0.001-G2RPO-surp-1e-06-conf-1e-06-horizon-1-decay-0.5/best_steps.txt"
with open(best_steps_file, "r") as f:
    ckpt_numbers=[int(line.strip()) for line in f.readlines()]

print(ckpt_numbers)


## gather eval_results for ablation_study

In [ ]:
from pathlib import Path

datas=['MATH-500', 'GSM8K', 'DAPO-Math-17k']
surps=['0', '1e-6']
confs=['0', '1e-6']

def fw_eval_results(base, surp, conf, data, output):
    p = Path(base)
    last_row = None
    rows = []
    for path in p.rglob("eval_results.csv"):
        with path.open(newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                if row and any(cell.strip() for cell in row):
                    last_row = row
        if last_row is None:
            continue
        if last_row is not None:
            rows.append([data, surp, conf] + last_row)
    with output.open("a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(rows)
        

for surp in surps:
    for conf in confs:
        for data in datas:
            if surp == '0' and conf == '0':
                pass
            else:
                if surp == '1e-6' and conf == '1e-6':
                    base = f"Qwen2.5-0.5B-Instruct_{data}_G2RPO_last_token"
                else:
                    base = f"Qwen2.5-0.5B-Instruct_{data}_G2RPO_last_token_surp-{surp}_conf-{conf}"
                fw_eval_results(base, surp=surp, conf=conf, data=data, output=Path('./ablation.csv'))
 

In [ ]:
from pathlib import Path
import csv
datas=['MATH-500', 'GSM8K', 'DAPO-Math-17k']
def fw_eval_results(base, surp, conf, msz, data, output, mode):
    p = Path(base)
    last_row = None
    rows = []
    for path in p.rglob("eval_results.csv"):
        with path.open(newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                if row and any(cell.strip() for cell in row):
                    last_row = row
        if last_row is None:
            continue
        if last_row is not None:
            rows.append([msz, data, mode] + last_row)
    with output.open("a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(rows)
        
best_surp_conf={'0.5':{
                        'GSM8K': ['1','1'],
                        'MATH-500': ['1', '1e-6'],
                        'DAPO-Math-17k': ['1e-6', '1e-2']
                      },
                '1.5':{
                        'GSM8K': ['1', '1'],
                        'MATH-500': ['1e-6','1e-6'],
                        'DAPO-Math-17k': ['1e-2', '1e-4']
                      },
                '7':{
                        'GSM8K': ['1', '1e-6'],
                        'MATH-500': ['1e-6', '1e-6'],
                        'DAPO-Math-17k': ['1e-6', '1e-6']
                }
                }

for data in datas:
    for msz in ['0.5', '1.5', '7']:
        # for NI, we try both
        base = f"Qwen2.5-{msz}B-Instruct_{data}_G2RPO_last_token_surp-0_conf-1e-6"
        fw_eval_results(base, surp='0', conf='1e-6', msz=msz, data=data, output=Path('./ablation.csv'), mode='ni')
        base = f"Qwen2.5-{msz}B-Instruct_{data}_G2RPO_next_token_surp-0_conf-1e-6"
        fw_eval_results(base, surp='0', conf='1e-6', msz=msz, data=data, output=Path('./ablation.csv'), mode='ni')

        base = f"Qwen2.5-{msz}B-Instruct_{data}_G2RPO_next_token"
        fw_eval_results(base, surp='1e-6', conf='1e-6', msz=msz, data=data, output=Path('./ablation.csv'), mode='ne')
        if best_surp_conf[msz][data]!=['1e-6', '1e-6']:
            base = f"Qwen2.5-{msz}B-Instruct_{data}_G2RPO_last_token_surp-{best_surp_conf[msz][data][0]}_conf-{best_surp_conf[msz][data][1]}"
        else:
            base = f"Qwen2.5-{msz}B-Instruct_{data}_G2RPO_last_token"
        fw_eval_results(base, surp='1e-6', conf='1e-6', msz=msz, data=data, output=Path('./ablation.csv'), mode='vanilla')
        

## Gather eval_results for parameter analysis

In [ ]:
from pathlib import Path
import csv 

datas=['MATH-500', 'GSM8K', 'DAPO-Math-17k']
surps=['1e-6', '1e-4', '1e-2', '1']
confs=['1e-6', '1e-4', '1e-2', '1']

def fw_eval_results(base, surp, conf, data, output):
    p = Path(base)
    last_row = None
    rows = []
    for path in p.rglob("eval_results.csv"):
        with path.open(newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                if row and any(cell.strip() for cell in row):
                    last_row = row
        if last_row is None:
            continue
        if last_row is not None:
            rows.append([data, surp, conf] + last_row)
    with output.open("a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(rows)
        
msz='1.5B'
for surp in surps:
    conf='1e-6'
    for data in datas:
        if surp == '1e-6' and conf == '1e-6':
            # pass
            base = f"Qwen2.5-{msz}-Instruct_{data}_G2RPO_last_token"
        else:
            base = f"Qwen2.5-{msz}-Instruct_{data}_G2RPO_last_token_surp-{surp}_conf-{conf}"
        fw_eval_results(base, surp=surp, conf=conf, data=data, output=Path(f'./param_ana_{msz}.csv'))

for conf in confs:
    surp='1e-6'
    for data in datas:
        if surp == '1e-6' and conf == '1e-6':
            pass
            # base = f"Qwen2.5-{msz}-Instruct_{data}_G2RPO_last_token"
        else:
            base = f"Qwen2.5-{msz}-Instruct_{data}_G2RPO_last_token_surp-{surp}_conf-{conf}"
            fw_eval_results(base, surp=surp, conf=conf, data=data, output=Path(f'./param_ana_{msz}.csv'))
 

## Gather results for fine-grained eval results for param analysis

In [ ]:
from pathlib import Path
import csv
datas=['MATH-500', 'GSM8K', 'DAPO-Math-17k']
# datas=['GSM8K']
surps=['1e-6', '1e-5', '1e-4', '1e-3' ,'1e-2', '1e-1', '1']
confs=['1e-6', '1e-5', '1e-4', '1e-3' ,'1e-2', '1e-1', '1']

def fw_eval_results(base, surp, conf, data, output):
    p = Path(base)
    last_row = None
    rows = []
    for path in p.rglob("eval_results.csv"):
        with path.open(newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                if row and any(cell.strip() for cell in row):
                    last_row = row
        if last_row is None:
            continue
        if last_row is not None:
            rows.append([data, surp, conf] + last_row)
    with output.open("a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(rows)
        

model_size='1.5B'
for surp in surps:
    for conf in confs:
        for data in datas:
            if surp == '1e-6' and conf == '1e-6':
                pass
            else:
                base = f"Qwen2.5-{model_size}-Instruct_{data}_G2RPO_last_token_surp-{surp}_conf-{conf}"
                fw_eval_results(base, surp=surp, conf=conf, data=data, output=Path(f'./param_ana_{model_size}.csv'))
 

## Case Study

In [ ]:
# import os
# def read_and_write_for_case_study(run_path, write_csv_path, name):
#     # TODO: find last row in os.join(run_path, "eval_results.csv")'s first value as ckpt
#     ckpt_folder="checkpoint-"+str(ckpt)
#     eval_results_text_path=os.path.join(run_path, ckpt_folder, "eval_results_text.csv")
#     # TODO: read csv from eval_results_text_path
#     # TODO: write in the new columns to write_csv_path, if there exist things, join the column "prompt". All other headers add {name}_xxx

# def get_results_from_project(base):
#     write_csv_path="./case_study.csv"
#     folders = os.listdir(base)
#     for run_folder in folders:
#         run_path = os.path.join(base, run_folder)
#         if os.isdir(run_path):
#             method = run_folder.split('-')[-1]
#             read_and_write_csv(run_path, write_csv_path, name=method)
        

# base = "./Qwen2.5-7B-Instruct_GSM8K"
# get_results_from_project(base)
# base = "./Qwen2.5-7B-Instruct_GSM8K_G2RPO_last_token_surp-1_conf-1e-6"
# get_results_from_project(base)

        
import os
import pandas as pd

def read_and_write_for_case_study(run_path, write_csv_path, name):
    # find last row in eval_results.csv's first value as ckpt
    eval_results_path = os.path.join(run_path, "eval_results.csv")
    df_eval = pd.read_csv(eval_results_path)
    ckpt = df_eval.iloc[-1, 0]
    ckpt_folder = "checkpoint-" + str(ckpt)
    eval_results_text_path = os.path.join(run_path, ckpt_folder, "eval_results_text.csv")
    
    df_text = pd.read_csv(eval_results_text_path)
    # write in the new columns to write_csv_path:
    # keep 'prompt' as-is; all other headers prefixed with {name}_xxx
    rename_map = {col: (col if col == "prompt" else f"{name}_{col}") for col in df_text.columns}
    df_new = df_text.rename(columns=rename_map)

    # If write_csv_path exists, merge on 'prompt'; else create
    if os.path.isfile(write_csv_path):
        df_out = pd.read_csv(write_csv_path)
        if not df_out.empty and "prompt" in df_out.columns:
            df_merged = pd.merge(df_out, df_new, on="prompt", how="outer")
        else:
            df_merged = df_new
    else:
        df_merged = df_new

    # Write the merged results back
    os.makedirs(os.path.dirname(write_csv_path), exist_ok=True)
    df_merged.to_csv(write_csv_path, index=False)
    print(f"Wrote merged results to {write_csv_path} for method '{name}'")

def get_results_from_project(base):
    write_csv_path = "./case_study.csv"
    if not os.path.isdir(base):
        print(f"Base not found or not a directory: {base}")
        return

    folders = os.listdir(base)
    for run_folder in folders:
        run_path = os.path.join(base, run_folder)
        if os.path.isdir(run_path):
            method = run_folder.split('-')[-1]
            read_and_write_for_case_study(run_path, write_csv_path, name=method)

# Run over projects
base = "./Qwen2.5-7B-Instruct_GSM8K"
get_results_from_project(base)
base = "./Qwen2.5-7B-Instruct_GSM8K_G2RPO_last_token_surp-1_conf-1e-6"
get_results_from_project(base)


In [ ]:
import os
import pandas as pd

def read_and_write_for_case_study(run_path, write_csv_path, name):
    # find last row in eval_results.csv's first value as ckpt
    eval_results_path = os.path.join(run_path, "eval_results.csv")
    df_eval = pd.read_csv(eval_results_path)
    ckpt = df_eval.iloc[-1, 0]
    ckpt_folder = "checkpoint-" + str(ckpt)
    eval_results_text_path = os.path.join(run_path, ckpt_folder, "eval_results_text.csv")

    df_text = pd.read_csv(eval_results_text_path)
    # keep 'prompt' as-is; all other headers prefixed with {name}_xxx
    rename_map = {col: (col if col == "prompt" else f"{name}_{col}") for col in df_text.columns}
    df_new = df_text.rename(columns=rename_map)

    # Append new columns (no merge); keep 'prompt' only once
    if os.path.isfile(write_csv_path):
        df_out = pd.read_csv(write_csv_path)
        if "prompt" in df_out.columns:
            df_append = df_new.drop(columns=["prompt"], errors="ignore")
        else:
            df_append = df_new
        df_merged = pd.concat([df_out, df_append], axis=1)
    else:
        df_merged = df_new

    # Write the results back
    os.makedirs(os.path.dirname(write_csv_path), exist_ok=True)
    df_merged.to_csv(write_csv_path, index=False)
    print(f"Wrote appended columns to {write_csv_path} for method '{name}'")

def get_results_from_project(base):
    write_csv_path = "./case_study.csv"

    folders = os.listdir(base)
    for run_folder in folders:
        run_path = os.path.join(base, run_folder)
        if os.path.isdir(run_path):
            method = run_folder.split('-')[-1]
            read_and_write_for_case_study(run_path, write_csv_path, name=method)

# Run over projects
base = "./Qwen2.5-7B-Instruct_GSM8K"
get_results_from_project(base)
base = "./Qwen2.5-7B-Instruct_GSM8K_G2RPO_last_token_surp-1_conf-1e-6"
get_results_from_project(base)

## Collect all the trainer_states.json for big table


In [ ]:
def model_data_extracter(proj):
    sizes=['0.5B', '1.5B', '7B']
    for size in sizes:
        if size in proj:
            model_size=size
    datas=['GSM8K','MATH-500','DAPO-Math-17k'] 
    for data in datas:
        if data in proj:
            data_proj=data
    return model_size, data_proj

import re
def find_max_ckpt(names):
    steps = []
    for name in names:
        m = re.fullmatch(r'checkpoint-(\d+)', name)
        if m:
            steps.append(int(m.group(1)))
    return max(steps)

In [ ]:
# copy trainer_states.json for baselines
import shutil
import os

base_path='./'
trainer_states_path_to_save = os.path.join(base_path, 'trainer_states')
if not os.path.exists(trainer_states_path_to_save):
    os.makedirs(trainer_states_path_to_save, exist_ok=True)

# baseline_projs=['qwen2.5-0.5B_GSM8K_nov_30_reward_rectified', 'qwen2.5-0.5B_MATH-500_nov_20_reward_rectified', 'Qwen2.5-0.5B-Instruct_DAPO-Math-17k', 
#                 'qwen2.5-1.5B_GSM8K_nov_30_reward_rectified', 'qwen2.5-1.5B_MATH-500_nov_30_reward_rectified', 'Qwen2.5-1.5B-Instruct_DAPO-Math-17k',
#                 'Qwen2.5-7B-Instruct_GSM8K', 'Qwen2.5-7B-Instruct_MATH-500', 'Qwen2.5-7B-Instruct_DAPO-Math-17k']
baseline_projs=['Qwen2.5-0.5B-Instruct_GSM8K', 'Qwen2.5-0.5B-Instruct_MATH-500', 'Qwen2.5-0.5B-Instruct_DAPO-Math-17k',
                'Qwen2.5-1.5B-Instruct_GSM8K', 'Qwen2.5-1.5B-Instruct_MATH-500', 'Qwen2.5-1.5B-Instruct_DAPO-Math-17k',
                'Qwen2.5-7B-Instruct_GSM8K', 'Qwen2.5-7B-Instruct_MATH-500', 'Qwen2.5-7B-Instruct_DAPO-Math-17k']
# baseline_methods=['DAPO', 'GFPO', 'GTPO', 'SGRPOLee']
baseline_methods=['GTPO']

for proj in baseline_projs:
    for method in baseline_methods:
        model_size, data = model_data_extracter(proj)
        run_folder=f'-shared-public-models-Qwen-Qwen2.5-{model_size}-Instruct-{data}-kl-0.001-{method}'
        big_table_runs_path = os.path.join(base_path, proj, run_folder)
        # take the last checkpoint
        ckpts = os.listdir(big_table_runs_path)
        max_ckpt_num = find_max_ckpt(ckpts)
        max_ckpt_trainer_state_path=os.path.join(big_table_runs_path, 'checkpoint-'+str(max_ckpt_num), 'trainer_state.json')
        
        source_file=max_ckpt_trainer_state_path
        dest_file=os.path.join(trainer_states_path_to_save, f'{model_size}_{data}_{method}_trainer_states.json')
        shutil.copy2(source_file, dest_file)
        print(f"'{source_file}' copied to '{dest_file}' successfully.")

In [ ]:
# copy trainer_states.json for our method
import shutil
import os

base_path='./'
trainer_states_path_to_save = os.path.join(base_path, 'trainer_states')
if not os.path.exists(trainer_states_path_to_save):
    os.makedirs(trainer_states_path_to_save, exist_ok=True)

datas=['GSM8K', 'MATH-500', 'DAPO-Math-17k']
models=['Qwen2.5-0.5B-Instruct', 'Qwen2.5-1.5B-Instruct']
baseline_projs=[model+'_'+data for model in models for data in datas]
baseline_methods=['DAPO', 'GFPO', 'GTPO', 'SGRPOLee']


for proj in baseline_projs:
    for method in baseline_methods:
        model_size, data = model_data_extracter(proj)
        run_folder=f'-shared-public-models-Qwen-Qwen2.5-{model_size}-Instruct-{data}-kl-0.001-{method}'
        big_table_runs_path = os.path.join(base_path, proj, run_folder)
        # take the last checkpoint
        ckpts = os.listdir(big_table_runs_path)
        max_ckpt_num = find_max_ckpt(ckpts)
        max_ckpt_trainer_state_path=os.path.join(big_table_runs_path, 'checkpoint-'+str(max_ckpt_num), 'trainer_state.json')
        
        source_file=max_ckpt_trainer_state_path
        dest_file=os.path.join(trainer_states_path_to_save, f'{model_size}_{data}_{method}_trainer_states.json')
        shutil.copy2(source_file, dest_file)
        print(f"'{source_file}' copied to '{dest_file}' successfully.")

## Draw the curves based on trainer_state.json

In [ ]:
import os
def craft_paths_dict(model_size, data):
    base='./trainer_states'
    paths_dict={}
    for method in ['DAPO', 'GFPO', 'GTPO', 'S-GRPO', 'IAPO']:
        ts_path=os.path.join(base,f'{model_size}_{data}_{method}_trainer_states.json')
        if method == 'IAPO':
            ts_path=os.path.join(base,f'{model_size}_{data}_G2RPO_trainer_states.json')
        if method == 'S-GRPO':
            ts_path=os.path.join(base,f'{model_size}_{data}_SGRPOLee_trainer_states.json')
        paths_dict[method]=ts_path
    return paths_dict

In [ ]:
# import json
# import matplotlib.pyplot as plt

# def plot_curve(paths, plot_dest, metric="rewards/correctness_reward_func", y_label='Correctness reward', with_std=True):
#     """
#     paths: dict[str, str] where key = method name, value = path to JSON file
#     """
#     plt.figure(figsize=(10, 5))
#     colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [
#         "tab:blue", "tab:orange", "tab:green", "tab:red",
#         "tab:purple", "tab:brown", "tab:pink", "tab:gray",
#         "tab:olive", "tab:cyan"
#     ])

#     for i, (method, path) in enumerate(paths.items()):
#         # Load JSON
#         with open(path, "r") as f:
#             data = json.load(f)

#         # Extract entries with metrics
#         print(f"[DEBUG]: {metric}")
#         metric_mean=metric+"/mean"
#         entries = [d for d in data.get("log_history", [])
#                    if "step" in d and metric_mean in d]
#         if entries != []:
#             print("[DEBUG]: Entries nonempty!!")
#         if len(entries) == 0:
#             entries = [d for d in data.get("log_history", [])
#                    if "step" in d and metric in d]
#             with_std=False
#             metric_mean=metric


#         if not entries:
#             continue
#         entries.sort(key=lambda x: x["step"])

#         steps = [e["step"] for e in entries]
#         means = [e[metric_mean] for e in entries]
        

#         color = colors[i % len(colors)]
#         plt.plot(steps, means, color=color, label=method)
#         if with_std==True:
#             stds  = [e.get(metric+"/std", 0.0) for e in entries]
#             lower = [m - s for m, s in zip(means, stds)]
#             upper = [m + s for m, s in zip(means, stds)]
#             plt.fill_between(steps, lower, upper, color=color, alpha=0.2)

#     plt.xlabel("Step")
#     plt.ylabel(y_label)
#     # plt.title("Correctness Reward vs Steps (±1 std)")
#     plt.grid(True, alpha=0.3)
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(plot_dest)
#     # plt.show()


import os
import json
import matplotlib.pyplot as plt

def plot_curve(paths, ax, metric="rewards/correctness_reward_func", y_label='Correctness reward', with_std=True):
    """
    paths: dict[str, str] where key = method name, value = path to JSON file
    ax: matplotlib Axes to plot on
    """
    colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [
        "tab:blue", "tab:orange", "tab:green", "tab:red",
        "tab:purple", "tab:brown", "tab:pink", "tab:gray",
        "tab:olive", "tab:cyan"
    ])

    for i, (method, path) in enumerate(paths.items()):
        # Load JSON
        with open(path, "r") as f:
            data = json.load(f)

        # Extract entries with metrics
        metric_mean = metric + "/mean"
        entries = [d for d in data.get("log_history", []) if "step" in d and metric_mean in d]
        if entries == []:
            # Fallback to raw metric if mean not present
            entries = [d for d in data.get("log_history", []) if "step" in d and metric in d]
            with_std = False
            metric_mean = metric

        if not entries:
            continue

        entries.sort(key=lambda x: x["step"])
        steps = [e["step"] for e in entries]
        means = [e[metric_mean] for e in entries]

        color = colors[i % len(colors)]
        ax.plot(steps, means, color=color, label=method)
        if with_std:
            stds = [e.get(metric + "/std", 0.0) for e in entries]
            lower = [m - s for m, s in zip(means, stds)]
            upper = [m + s for m, s in zip(means, stds)]
            ax.fill_between(steps, lower, upper, color=color, alpha=0.2)

    ax.set_xlabel("Step")
    ax.set_ylabel(y_label)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

In [ ]:
# def plot_all_for_metric(metric, y_label, with_std):
#     plot_dir='./trainer_states_plots'
#     for model_size in ['0.5B', '1.5B', '7B']:
#         for data in ['GSM8K', 'MATH-500', 'DAPO-Math-17k']:
#             paths_dict = craft_paths_dict(model_size, data)
#             plot_postfix=metric.replace('/','_')
#             plot_dest=os.path.join(plot_dir,f'{model_size}_{data}_{plot_postfix}.pdf')
#             plot_curve(paths_dict, plot_dest, metric, y_label, with_std)

def plot_all_for_metric(metric, y_label, with_std):
    plot_dir = './trainer_states_plots'
    plt.rcParams["font.family"] = "serif"
    # Specify 'Times New Roman' as the preferred serif font
    plt.rcParams["font.serif"] = ["Times New Roman"]
    fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(12, 10), sharex=False, sharey=False)
    
    model_sizes = ['0.5B', '1.5B', '7B']
    datasets = ['GSM8K', 'MATH-500', 'DAPO-Math-17k']

    axes = axs.ravel()
    idx = 0
    for ms in model_sizes:
        for ds in datasets:
            ax = axes[idx]
            paths_dict = craft_paths_dict(ms, ds)
            plot_curve(paths_dict, ax, metric=metric, y_label=y_label, with_std=with_std)
            ax.set_title(f"{ms} • {ds}", fontsize=10)
            idx += 1

    fig.tight_layout()
    plot_postfix = metric.replace('/', '_')
    out_path = os.path.join(plot_dir, f'grid_{plot_postfix}.pdf')
    fig.savefig(out_path)

In [ ]:
metrics_dict_of_interst = {'Correctness Reward': 'rewards/correctness_reward_func', 
                      'Mean Completion Lengths':'completions/mean_length'}
for metric_name, metric_handle in metrics_dict_of_interst.items():
    print(f"[DEBUG]: {metric_name}")
    plot_all_for_metric(metric_handle, metric_name, with_std=True)


In [ ]:
# import json
# import matplotlib.pyplot as plt

# def plot_curve_ratio(paths, plot_dest, metric_1="rewards/correctness_reward_func/mean", metric_2="completions/mean_length", y_label='Ratio', with_std=False):
#     """
#     paths: dict[str, str] where key = method name, value = path to JSON file
#     """
#     plt.figure(figsize=(10, 5))
#     colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [
#         "tab:blue", "tab:orange", "tab:green", "tab:red",
#         "tab:purple", "tab:brown", "tab:pink", "tab:gray",
#         "tab:olive", "tab:cyan"
#     ])

#     for i, (method, path) in enumerate(paths.items()):
#         # Load JSON
#         with open(path, "r") as f:
#             data = json.load(f)

#         # Extract entries with metrics
#         entries = [d for d in data.get("log_history", [])
#                    if "step" in d and metric_1 in d and metric_2 in d]
        

#         # entries = [entries_1[i]/entries_2[i] for i in range(len(entries_1))]
#         if len(entries) == 0:
#             entries = [d for d in data.get("log_history", [])
#                    if "step" in d and metric in d]
#             with_std=False
#             metric_mean=metric

#         if not entries:
#             continue
#         entries.sort(key=lambda x: x["step"])

#         steps = [e["step"] for e in entries]
#         means = [e[metric_1]/e[metric_2] for e in entries]
        

#         color = colors[i % len(colors)]
#         plt.plot(steps, means, color=color, label=method)
#         if with_std==True:
#             stds  = [e.get(metric+"/std", 0.0) for e in entries]
#             lower = [m - s for m, s in zip(means, stds)]
#             upper = [m + s for m, s in zip(means, stds)]
#             plt.fill_between(steps, lower, upper, color=color, alpha=0.2)

#     plt.xlabel("Step")
#     plt.ylabel(y_label)
#     # plt.title("Correctness Reward vs Steps (±1 std)")
#     plt.grid(True, alpha=0.3)
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(plot_dest)
#     # plt.show()

import os
import json
import matplotlib.pyplot as plt

def _base_metric(metric):
    return metric[:-5] if metric.endswith('/mean') else metric

def plot_curve_ratio(paths, ax,
                     metric_1="rewards/correctness_reward_func/mean",
                     metric_2="completions/mean_length",
                     y_label='Ratio', with_std=False):
    """
    paths: dict[str, str] where key = method name, value = path to JSON file
    ax: matplotlib Axes to plot on
    """
    colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [
        "tab:blue", "tab:orange", "tab:green", "tab:red",
        "tab:purple", "tab:brown", "tab:pink", "tab:gray",
        "tab:olive", "tab:cyan"
    ])

    m1_mean = metric_1
    m2_mean = metric_2
    m1_std  = _base_metric(metric_1) + "/std"
    m2_std  = _base_metric(metric_2) + "/std"

    for i, (method, path) in enumerate(paths.items()):
        with open(path, "r") as f:
            data = json.load(f)

        entries = [d for d in data.get("log_history", [])
                   if "step" in d and m1_mean in d and m2_mean in d]

        # Fallback: use base metrics if '/mean' keys not present
        if len(entries) == 0:
            m1_mean = _base_metric(metric_1)
            m2_mean = _base_metric(metric_2)
            entries = [d for d in data.get("log_history", [])
                       if "step" in d and m1_mean in d and m2_mean in d]
            with_std = False  # no std band without explicit std keys

        if not entries:
            continue

        entries.sort(key=lambda x: x["step"])
        steps, means = [], []
        lowers, uppers = [], []

        for e in entries:
            a = e[m1_mean]
            b = e[m2_mean]
            if b == 0:
                continue
            r = a / b
            steps.append(e["step"])
            means.append(r)

            if with_std:
                sa = e.get(m1_std, None)
                sb = e.get(m2_std, None)
                if sa is not None and sb is not None and b != 0:
                    # Error propagation for ratio r = a/b (ignoring covariance)
                    # σ_r^2 ≈ (σ_a/b)^2 + (a*σ_b/b^2)^2
                    var_r = (sa / b) ** 2 + ((a * sb) / (b ** 2)) ** 2
                    sr = var_r ** 0.5
                    lowers.append(r - sr)
                    uppers.append(r + sr)
                else:
                    with_std = False  # disable band if stds unavailable

        color = colors[i % len(colors)]
        ax.plot(steps, means, color=color, label=method)
        if with_std and lowers and uppers:
            ax.fill_between(steps, lowers, uppers, color=color, alpha=0.2)

    ax.set_xlabel("Step")
    ax.set_ylabel(y_label)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

In [ ]:
# def plot_all_for_ratio():
#     plot_dir='./trainer_states_plots'
#     for model_size in ['0.5B', '1.5B', '7B']:
#         for data in ['GSM8K', 'MATH-500', 'DAPO-Math-17k']:
#             paths_dict = craft_paths_dict(model_size, data)
#             # plot_postfix=metric.replace('/','_')
#             plot_dest=os.path.join(plot_dir,f'{model_size}_{data}_Ratio.pdf')
#             plot_curve_ratio(paths_dict, plot_dest)
def plot_all_for_ratio():
    plot_dir = './trainer_states_plots'
    fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(12, 10), sharex=False, sharey=False)

    model_sizes = ['0.5B', '1.5B', '7B']
    datasets = ['GSM8K', 'MATH-500', 'DAPO-Math-17k']

    axes = axs.ravel()
    idx = 0
    for ms in model_sizes:
        for ds in datasets:
            ax = axes[idx]
            paths_dict = craft_paths_dict(ms, ds)
            plot_curve_ratio(paths_dict, ax,
                             metric_1="rewards/correctness_reward_func/mean",
                             metric_2="completions/mean_length",
                             y_label='Ratio', with_std=False)
            ax.set_title(f"{ms} • {ds}", fontsize=10)
            idx += 1

    fig.tight_layout()
    out_path = os.path.join(plot_dir, 'grid_Ratio.pdf')
    fig.savefig(out_path)
    # plt.show()
plot_all_for_ratio()


## Collect wall-clock time for reasoning

In [ ]:
import os

base='./'
prefix='-shared-public-models-Qwen'
postfix='surp-1e-06-conf-1e-06-horizon-1-decay-0.5'
model='Qwen2.5-7B-Instruct'

datas=['GSM8K', 'MATH-500', 'DAPO-Math-17k']
methods=['DAPO', 'GFPO', 'GTPO', 'SGRPOLee', 'G2RPO', 'untrained']

for data in datas:
    for method in methods:
        try:
            if method == 'G2RPO':
                folder=os.path.join(base, f'{model}_{data}_{method}_last_token', f'{prefix}-{model}-{data}-kl-0.001-{method}-{postfix}')
            elif method == 'untrained':
                folder=f'./untrained_model_eval/{prefix}-{model}/{data}'
            else:
                folder=os.path.join(base, f'{model}_{data}', f'{prefix}-{model}-{data}-kl-0.001-{method}')
                
            # read 'eval_results.csv', last column last row has the wall-clock time
            with open(os.path.join(folder, 'eval_results.csv'), 'r') as f:
                lines = f.readlines()
                last_line = lines[-1]
                last_value = last_line.strip().split(',')[-1]
                print(f'{data} {method} time: {last_value} seconds')
            # Write to 'wall_clock_time.csv'
            with open('wall_clock_time.csv', 'a') as f:
                f.write(f'{data},{method},{last_value}\n')
        except:
            print('Failed to retrieve:', 'data:', data, 'method', method)


        

## Collect Results for CommonsenseQA

In [ ]:
import os

base='./'
prefix='-shared-public-models-Qwen'
model='Qwen2.5-0.5B-Instruct'

datas=['CommonsenseQA']
methods=['DAPO', 'GFPO', 'GTPO', 'SGRPOLee', 'G2RPO', 'untrained']


def open_folder_and_write(folder, data, method, surp, conf):
    with open(os.path.join(folder, 'eval_results.csv'), 'r') as f:
        lines = f.readlines()
        last_line = lines[-1]
        # last_value = last_line.strip().split(',')[-1]
        # print(f'{data} {method} time: {last_value} seconds')
    # Write to 'wall_clock_time.csv'
    with open('commensenseqa_results.csv', 'a') as f:
        f.write(f'{data},{method},{surp}, {conf}, {last_line}\n')

num_trans={'1e-6':'1e-06', '1e-4':'0.0001', '1e-2':'0.01', '1':'1.0'}
for data in datas:
    for method in methods:
        if method == 'G2RPO':
            for surp in ['1e-4','1e-2','1']:
                conf='1e-6'
                postfix=f'surp-{num_trans[surp]}-conf-{num_trans[conf]}-horizon-1-decay-0.5'
                folder=os.path.join(base, f'{model}_{data}_{method}_last_token_surp-{surp}_conf-{conf}', f'{prefix}-{model}-{data}-kl-0.001-{method}-{postfix}')
                open_folder_and_write(folder, data, method, surp, conf)
            for conf in ['1e-4','1e-2','1']:
                surp='1e-6'
                postfix=f'surp-{num_trans[surp]}-conf-{num_trans[conf]}-horizon-1-decay-0.5'
                folder=os.path.join(base, f'{model}_{data}_{method}_last_token_surp-{surp}_conf-{conf}', f'{prefix}-{model}-{data}-kl-0.001-{method}-{postfix}')
                open_folder_and_write(folder, data, method, surp, conf)
        elif method == 'untrained':
            folder=f'./untrained_model_eval/{prefix}-{model}/{data}'
            open_folder_and_write(folder, data, method, None, None)
        else:
            folder=os.path.join(base, f'{model}_{data}', f'{prefix}-{model}-{data}-kl-0.001-{method}') 
            # read 'eval_results.csv', last column last row has the wall-clock time
            open_folder_and_write(folder, data, method, None, None)



        

## Get newly trained GTPO results

In [ ]:
import os

base='./'
prefix='-shared-public-models-Qwen'
models=['Qwen2.5-0.5B-Instruct', 'Qwen2.5-1.5B-Instruct', 'Qwen2.5-7B-Instruct']

datas=['GSM8K', 'MATH-500', 'DAPO-Math-17k']
methods=['GTPO']


def open_folder_and_write(folder, data, model, surp, conf):
    with open(os.path.join(folder, 'eval_results.csv'), 'r') as f:
        lines = f.readlines()
        last_line = lines[-1]
        # last_value = last_line.strip().split(',')[-1]
        # print(f'{data} {method} time: {last_value} seconds')
    # Write to 'wall_clock_time.csv'
    with open('new_gtpo_results.csv', 'a') as f:
        f.write(f'{data},{model},{surp}, {conf}, {last_line}')

num_trans={'1e-6':'1e-06', '1e-4':'0.0001', '1e-2':'0.01', '1':'1.0'}
for model in models:
    for data in datas:
        folder=os.path.join(base, f'{model}_{data}', f'{prefix}-{model}-{data}-kl-0.001-GTPO') 
        # read 'eval_results.csv', last column last row has the wall-clock time
        open_folder_and_write(folder, data, model, None, None)

## Collect all final checkpoints

In [ ]:
import shutil
import os

ckpts={"./Qwen2.5-0.5B-Instruct_DAPO-Math-17k_G2RPO_last_token_surp-1e-6_conf-1e-2/-shared-public-models-Qwen-Qwen2.5-0.5B-Instruct-DAPO-Math-17k-kl-0.001-G2RPO-surp-1e-06-conf-0.01-horizon-1-decay-0.5/checkpoint-500": "Qwen2.5-0.5B-Instruct_DAPO-Math-17k", 
      "./Qwen2.5-0.5B-Instruct_GSM8K_G2RPO_last_token_surp-1_conf-1/-shared-public-models-Qwen-Qwen2.5-0.5B-Instruct-GSM8K-kl-0.001-G2RPO-surp-1.0-conf-1.0-horizon-1-decay-0.5/checkpoint-1100": "Qwen2.5-0.5B-Instruct_GSM8K",
      "./Qwen2.5-0.5B-Instruct_MATH-500_G2RPO_last_token_surp-1_conf-1e-6/-shared-public-models-Qwen-Qwen2.5-0.5B-Instruct-MATH-500-kl-0.001-G2RPO-surp-1.0-conf-1e-06-horizon-1-decay-0.5/checkpoint-4200": "Qwen2.5-0.5B-Instruct_MATH-500",
      "./Qwen2.5-1.5B-Instruct_DAPO-Math-17k_G2RPO_last_token_surp-1e-2_conf-1e-4/-shared-public-models-Qwen-Qwen2.5-1.5B-Instruct-DAPO-Math-17k-kl-0.001-G2RPO-surp-0.01-conf-0.0001-horizon-1-decay-0.5/checkpoint-200": "Qwen2.5-1.5B-Instruct_DAPO-Math-17k",
      "./Qwen2.5-1.5B-Instruct_GSM8K_G2RPO_last_token_surp-1_conf-1/-shared-public-models-Qwen-Qwen2.5-1.5B-Instruct-GSM8K-kl-0.001-G2RPO-surp-1.0-conf-1.0-horizon-1-decay-0.5/checkpoint-1000": "Qwen2.5-1.5B-Instruct_GSM8K",
      "./Qwen2.5-1.5B-Instruct_MATH-500_G2RPO_last_token/-shared-public-models-Qwen-Qwen2.5-1.5B-Instruct-MATH-500-kl-0.001-G2RPO-surp-1e-06-conf-1e-06-horizon-1-decay-0.5/checkpoint-1900": "Qwen2.5-1.5B-Instruct_MATH-500", 
      "./Qwen2.5-7B-Instruct_DAPO-Math-17k_G2RPO_last_token/-shared-public-models-Qwen-Qwen2.5-7B-Instruct-DAPO-Math-17k-kl-0.001-G2RPO-surp-1e-06-conf-1e-06-horizon-1-decay-0.5/checkpoint-100": "Qwen2.5-7B-Instruct_DAPO-Math-17k",
      "./Qwen2.5-7B-Instruct_GSM8K_G2RPO_last_token_surp-1_conf-1e-6/-shared-public-models-Qwen-Qwen2.5-7B-Instruct-GSM8K-kl-0.001-G2RPO-surp-1.0-conf-1e-06-horizon-1-decay-0.5/checkpoint-400": "Qwen2.5-7B-Instruct_GSM8K",
      "./Qwen2.5-7B-Instruct_MATH-500_G2RPO_last_token/-shared-public-models-Qwen-Qwen2.5-7B-Instruct-MATH-500-kl-0.001-G2RPO-surp-1e-06-conf-1e-06-horizon-1-decay-0.5/checkpoint-300": "Qwen2.5-7B-Instruct_MATH-500"
}

for src, dst in ckpts.items():
      print("copying to", dst)
      dst = os.path.join("./iapo_final_ckpts", dst)
      shutil.copytree(src, dst, dirs_exist_ok=False)